# SparkOCR-VLM — Databricks Free Edition

Package pre-installed via job environment spec. Reads PDFs from Volume, runs OCR, writes to `workspace.default.ocr_silver`.

In [ ]:
import os

# Paste your OpenRouter key here (rotate after use — never commit a real key)
api_key = 'sk-or-v1-PASTE_YOUR_KEY_HERE'
os.environ['OPENROUTER_API_KEY'] = api_key

print('Spark:', spark.version)
print('Key set.')

In [ ]:
INPUT    = '/Volumes/workspace/default/sparkocr_demo/input/'
UC_TABLE = 'workspace.default.ocr_silver'

files = dbutils.fs.ls(INPUT)
print(f'PDFs: {len(files)}')
for f in files:
    print(f'  {f.name}  {f.size/1024:.1f} KB')

In [ ]:
from sparkocr_vlm import OCRPipeline

pipeline = OCRPipeline(
    backend='openrouter',
    model='nvidia/nemotron-nano-12b-v2-vl:free',
    input_path=INPUT,
    batch_size=1,
    max_cost_usd=0.50,
    # Key passed directly into the UDF closure — works on serverless
    # because it gets pickled with the UDF, no env var propagation needed
    backend_kwargs={'api_key': api_key},
)

print('Running OCR...')
silver = pipeline.run(spark)
print('Pages:', silver.count())

In [ ]:
(
    silver
    .write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(UC_TABLE)
)
print(f'Saved to {UC_TABLE}')

In [ ]:
display(spark.table(UC_TABLE).select(
    'filename','page_num','markdown','prompt_tokens','completion_tokens','cost_usd','error'
))

In [ ]:
%%sql
SELECT filename, page_num, LEFT(markdown,300) AS markdown_preview, cost_usd, error
FROM workspace.default.ocr_silver
ORDER BY filename, page_num

In [ ]:
import json
rows = spark.table(UC_TABLE).collect()
summary = {
    'uc_table': UC_TABLE,
    'total_pages': len(rows),
    'total_cost_usd': round(sum(r['cost_usd'] or 0 for r in rows), 6),
    'errors': [f"{r['filename']} p{r['page_num']}" for r in rows if r['error']],
    'pages': [{
        'file': r['filename'], 'page': r['page_num'],
        'markdown': (r['markdown'] or '')[:500]
    } for r in rows],
}
print(json.dumps(summary, indent=2))
dbutils.notebook.exit(json.dumps(summary))